# Phase 2 — LeNet-5 (1998)

합성곱(Convolution)과 풀링(Pooling)으로 **공간 구조를 보존**하는 학습.
Phase 1의 flat MLP 대비 파라미터 수와 정확도 비교, Sigmoid/Tanh → ReLU 전환 효과 확인.

- 모델: `src/dl_history/models/lenet5.py` (`Conv2d` / `AvgPool2d` / `Linear` 직접 조합)
- 학습 루프: `src/dl_history/train.py` (Phase 2+ 공용)
- 전체 실험 재현: `uv run python scripts/run_phase2.py`

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import torch

from dl_history.data import load_fashion_mnist_loaders
from dl_history.metrics import EpochHistory, GradientHistory
from dl_history.models.lenet5 import LeNet5
from dl_history.models.mlp_torch import FlatMLP
from dl_history.train import train_model
from dl_history.viz import plot_accuracy_curves, plot_loss_curves

torch.manual_seed(0)

## 1. 출력 크기 공식 검증 — `W_out = (W − F + 2P) / S + 1`

손으로 계산한 체인과 실제 텐서 크기가 일치하는지 확인한다.

In [ ]:
model = LeNet5(activation="relu")

x = torch.randn(2, 1, 28, 28)          # input 1×28×28
x = model.pool(model.activate(model.conv1(x)))  # F=5, P=2, S=1 → 28; pool → 14
x = model.pool(model.activate(model.conv2(x)))  # F=5, P=0, S=1 → 10; pool → 5
print("pool2 output:", tuple(x.shape))
assert x.shape[1:] == (16, 5, 5)
assert x.numel() // 2 == model.fc1.in_features == 400  # flatten dim

## 2. 레이어별 파라미터 수

LeNet-5 전체 파라미터는 **약 6.1만 개**. flat MLP(784-128-10, 약 10.2만 개)보다 적으면서
공간 구조를 활용한다.

In [ ]:
rows = [(name, tuple(p.shape), p.numel()) for name, p in model.named_parameters()]
for name, shape, n in rows:
    print(f"{name:12s} {str(shape):18s} {n:>7,}")
print(f"{'total':12s} {'':18s} {sum(n for *_, n in rows):>7,}")

## 3. Exp A — 활성화 함수 비교 (Sigmoid / Tanh / ReLU)

1998년 당시의 squashing 함수(sigmoid, tanh)와 ReLU를 동일 LeNet-5에서 학습시켜
수렴 속도 차이를 본다. (노트북 데모는 3 epoch, 전체 실험은 스크립트에서 10 epoch)

In [ ]:
EPOCHS = 3
loaders = load_fashion_mnist_loaders(batch_size=128)

histories = {}
for activation in ["sigmoid", "tanh", "relu"]:
    torch.manual_seed(0)
    model = LeNet5(activation=activation)
    history, sec = train_model(model, loaders, epochs=EPOCHS, lr=0.05, momentum=0.9)
    histories[activation] = history
    print(f"{activation:8s} final val acc = {history.val_acc[-1]:.4f}  ({sec:.1f}s/epoch)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label, hist in histories.items():
    ax.plot(hist.val_acc, label=label)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation accuracy")
ax.set_title("LeNet-5 activation comparison"); ax.grid(alpha=0.3); ax.legend()
plt.show()

## 4. Exp B — LeNet-5 vs flat MLP

같은 조건에서 flat MLP(Phase 1의 2층 baseline과 동일 형태)와 비교:
파라미터는 더 적지만 convolution 덕분에 정확도가 더 높다.

In [ ]:
for name, module in [("LeNet-5", LeNet5()), ("Flat MLP", FlatMLP())]:
    print(f"{name:10s} {sum(p.numel() for p in module.parameters()):>7,} params")

torch.manual_seed(0)
lenet_hist, _ = train_model(LeNet5(activation="relu"), loaders, epochs=EPOCHS, lr=0.05, momentum=0.9)
torch.manual_seed(0)
mlp_hist, _ = train_model(FlatMLP(activation="relu"), loaders, epochs=EPOCHS, lr=0.05, momentum=0.9)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lenet_hist.val_acc, label="LeNet-5")
ax.plot(mlp_hist.val_acc, label="Flat MLP")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation accuracy")
ax.set_title("LeNet-5 vs flat MLP"); ax.grid(alpha=0.3); ax.legend()
plt.show()

## 정리

- Conv + Pool은 파라미터를 줄이면서 공간 구조를 보존해 flat MLP보다 정확도가 높다.
- Sigmoid/Tanh보다 ReLU가 초기 수렴이 빠르다 — Phase 1의 vanishing gradient 관찰과 일맥.
- 다음 Phase 3(AlexNet)에서는 ReLU + Dropout + GPU 학습으로 확장한다.